# Natural Language to SQL — Client Benchmark

**What this notebook does:** Loads 13 databases from `db/` (db-2, db-3, db-6..db-16), each with 30 question–SQL pairs. You can run the SQL against PostgreSQL and see which queries pass or fail. Use this to train, evaluate, or prototype agents that answer data questions in plain English.

**Run cells in order.** Prerequisites: `psycopg2-binary`, `pandas` installed (e.g. `pip install -r requirements.txt`).

**To execute SQL:** Start PostgreSQL first with `./scripts/setup_docker.sh -a` (13 containers on ports 5436–5448). If you skip this, you can still load and inspect the data.

## 1. Setup — Paths and Config

Resolves the working directory and loads which databases to include (or exclude via `exclude_dbs.json`). Assumes `psycopg2-binary` and `pandas` are installed.

In [11]:
import json
from pathlib import Path

# ROOT = directory containing db/ (zip root when client/ is extracted)
ROOT = Path.cwd().resolve()
if (ROOT / "db").exists():
    pass  # already at repo root (client/)
elif (ROOT.parent / "db").exists():
    ROOT = ROOT.parent.resolve()  # e.g. running from client/doc/
elif (ROOT / "client" / "db").exists():
    ROOT = (ROOT / "client").resolve()  # full repo: use client/
else:
    ROOT = (ROOT / "..").resolve() if (ROOT / ".." / "db").exists() else ROOT
ROOT = ROOT.resolve()
CLIENT_DB = ROOT / "db"

# Inlined mount logic (self-contained; no external scripts)
DB_PORTS_START = 5436
# Two layouts: (1) Repo root: db-1→5436, db-2→5437, ... (2) Client-only: db-2→5436, db-3→5437, db-6→5438, ...
_CLIENT_DB_NUMS = list(range(1, 17))  # db-1..db-16 (includes db-1, db-4, db-5)
_DB_PORT_MAP_CLIENT = {n: DB_PORTS_START + i for i, n in enumerate(_CLIENT_DB_NUMS)}
_DB_PORT_MAP_REPO = {n: DB_PORTS_START + n - 1 for n in range(1, 17)}
_use_repo_layout = None  # set by Docker check cell

def get_pg_port(db_num: int) -> int:
    if _use_repo_layout is True:
        return _DB_PORT_MAP_REPO.get(db_num, DB_PORTS_START)
    return _DB_PORT_MAP_CLIENT.get(db_num, DB_PORTS_START)

def load_client_db(client_db_dir: Path, db_num: int) -> dict:
    """Load docs and queries for db-N. Returns {db, docs, queries}."""
    db_id = f"db-{db_num}"
    db_dir = client_db_dir / db_id
    if not db_dir.exists():
        return {"db": db_id, "docs": None, "queries": [], "error": "not found"}
    docs = None
    docs_path = db_dir / "DOCUMENTATION" / "README.md"
    if docs_path.exists():
        docs = docs_path.read_text(encoding="utf-8")
    queries = []
    queries_path = db_dir / "QUERIES" / "queries.json"
    if queries_path.exists():
        data = json.loads(queries_path.read_text(encoding="utf-8"))
        queries = data.get("queries", [])
    return {"db": db_id, "docs": docs, "queries": queries}

def get_bird_pairs(mounted: dict, db_num: int) -> list:
    """Return BIRD-style pairs for db-N. Only includes keys present in queries.json."""
    result = []
    for q in mounted.get("queries", []):
        sql = q.get("sql") or q.get("SQL") or ""
        pair = {"sql": sql}
        for key in ("number", "question", "description", "evidence", "expected_output", "normal_query", "complexity", "title"):
            if key in q:
                pair[key] = q[key]
        result.append(pair)
    return result

DB_NUMS = list(range(1, 17))

# Derive exclusion from config (no hardcoded db numbers)
EXCLUDE_CONFIG = ROOT / "exclude_dbs.json"
excluded_nums = []
if EXCLUDE_CONFIG.exists():
    try:
        cfg = json.loads(EXCLUDE_CONFIG.read_text(encoding="utf-8"))
        excluded_nums = cfg.get("exclude", [])
    except Exception:
        pass
DB_NUMS_TEST = [n for n in DB_NUMS if n not in excluded_nums]

print(f"ROOT: {ROOT}")
print(f"CLIENT_DB: {CLIENT_DB}")
print(f"Ports: {get_pg_port(1)}–{get_pg_port(16)}")

ROOT: /Users/machine/Documents/AQ/db/client
CLIENT_DB: /Users/machine/Documents/AQ/db/client/db
Ports: 5436–5448


## 2. Docker Check — Is PostgreSQL Running?

Verifies that Docker is available and that the 14 PostgreSQL containers (one per database) are up. If you see 14/14, you're ready to run queries.

In [12]:
import subprocess

def check_docker() -> bool:
    try:
        r = subprocess.run(["docker", "info"], capture_output=True, timeout=5)
        return r.returncode == 0
    except Exception:
        return False

def check_containers() -> dict:
    try:
        r = subprocess.run(
            ["docker", "ps", "--format", "{{.Names}}"],
            capture_output=True, text=True, timeout=5
        )
        names = [n.strip() for n in r.stdout.splitlines() if "postgres-db-" in n]
        return {n: True for n in names}
    except Exception:
        return {}

docker_ok = check_docker()
containers = check_containers() if docker_ok else {}
# Detect layout: repo root has postgres-db-1 (port 5436=db-1); client-only has db-2..db-16 (port 5436=db-2)
global _use_repo_layout
_use_repo_layout = "postgres-db-1" in containers if containers else False
expected = 16 if _use_repo_layout else 13
print(f"Docker: {'✓' if docker_ok else '✗'}")
print(f"PostgreSQL containers: {len(containers)}/{expected} ({'repo layout' if _use_repo_layout else 'client layout'})")

Docker: ✓
PostgreSQL containers: 13/13 (client layout)


## 3. Mount Client Databases — Load Docs and Queries

Reads each database's documentation and `queries.json` into memory. You'll see how many queries and whether docs are present for each db-N.

In [13]:
mounted = {n: load_client_db(CLIENT_DB, n) for n in DB_NUMS_TEST}

for n, m in mounted.items():
    qty = len(m.get("queries", []))
    docs_ok = "✓" if m.get("docs") else "✗"
    print(f"  {m['db']}: {qty} queries, docs={docs_ok}")

  db-2: 30 queries, docs=✓
  db-3: 30 queries, docs=✓
  db-6: 30 queries, docs=✓
  db-7: 30 queries, docs=✓
  db-8: 30 queries, docs=✓
  db-9: 30 queries, docs=✓
  db-10: 30 queries, docs=✓
  db-11: 30 queries, docs=✓
  db-12: 30 queries, docs=✓
  db-13: 30 queries, docs=✓
  db-14: 30 queries, docs=✓
  db-15: 30 queries, docs=✓
  db-16: 30 queries, docs=✓


## 4. Run SQL — Connect and Execute

Connects to PostgreSQL and runs queries. Below we show the first 3 question–SQL pairs for db-2 and demonstrate executing one query.

In [14]:
import psycopg2

def get_connection(db_num: int):
    conn = psycopg2.connect(
        host="localhost", port=get_pg_port(db_num),
        user="postgres", password="postgres", dbname=f"db{db_num}",
        connect_timeout=30
    )
    conn.set_session(autocommit=True)
    return conn

def run_sql(db_num: int, sql: str, limit: int = 50) -> tuple:
    """Execute SQL; adds LIMIT if not present."""
    try:
        conn = get_connection(db_num)
        cur = conn.cursor()
        cur.execute("SET statement_timeout = '600000'")  # 10 min max per query (db-16 flood risk needs more)
        q = sql.strip().rstrip(";")
        if "LIMIT" not in q.upper() and "FETCH" not in q.upper():
            q = f"{q} LIMIT {limit}"
        cur.execute(q)
        cols = [d[0] for d in cur.description] if cur.description else []
        rows = cur.fetchall()
        conn.close()
        return (rows, cols)
    except Exception as e:
        return (None, str(e))

# BIRD-style pairs (use first available db; db-2 may be excluded)
_first_db = next((n for n in DB_NUMS_TEST if n in mounted and mounted[n].get("queries")), None)
pairs = get_bird_pairs(mounted[_first_db], _first_db) if _first_db else []
for p in pairs[:3]:
    num = p.get("number", "?")
    qn = str(p.get("question", ""))[:60]
    print(f"Q{num}: {qn}...")

Q1: Can you show me how each employee's daily sales have been tr...
Q2: Can you break down monthly purchase behavior by customer? I ...
Q3: Show me daily performance quartiles for each employee — I wa...


## 5. Example — Run One Query and View Results

Runs the first query from db-2 and displays the result as a table. If you see rows and columns, the connection and query work.

In [15]:
db_num = next((n for n in DB_NUMS_TEST if n in mounted and mounted[n].get("queries")), None)
if db_num is None:
    print("No databases with queries. Run Mount cell (Cell 6) first.")
else:
    pair = get_bird_pairs(mounted[db_num], db_num)[0]
    rows, cols = run_sql(db_num, pair["sql"])
    if rows is not None:
        print(f"Rows: {len(rows)}, Cols: {cols}")
        try:
            import pandas as pd
            df = pd.DataFrame(rows, columns=cols)
            display(df.head())
        except Exception:
            print(rows[:5])
    else:
        print(f"Error: {cols}")

Rows: 100, Cols: ['analysis_date', 'employee_id', 'record_count', 'avg_value', 'min_value', 'max_value', 'median_value', 'stddev_value', 'above_avg_count', 'avg_rolling_7d']


,analysis_date,employee_id,record_count,avg_value,min_value,max_value,median_value,stddev_value,above_avg_count,avg_rolling_7d
0,2026-02-10,46,12,3008501.083333333333,9645,5251141,3594868.0,1718356.8531,8,3240593.107142857143
1,2026-02-10,29,12,2853168.333333333333,254630,5113488,3168835.5,1646573.7045,7,2820766.071428571429
2,2026-02-10,17,12,2457037.833333333333,169968,4834692,2192845.0,1879717.1917,6,2434716.702380952381
3,2026-02-10,4,12,2516696.083333333333,325532,5296536,2652558.0,1351707.8786,5,2486245.797619047619
4,2026-02-10,37,11,3158604.272727272727,327734,5120944,3891304.0,1836128.0571,7,3130036.688311688312


## 6. Run Tests Across All Databases

Runs all 30 queries per database in parallel (one task per query across all DBs), then collects and reports pass/fail. Requires Docker PostgreSQL to be running (`./scripts/setup_docker.sh -a`). Excluded databases (from `exclude_dbs.json`) are skipped.

In [16]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def _test_one_query(db_num: int, pair: dict, sql_limit: int) -> dict:
    """Run a single query; returns {db, q, ok, rows, error}."""
    import time
    _log_path = (ROOT.parent if (ROOT / "db").exists() else ROOT) / ".cursor" / "notebook-db-results.log"
    q_num = pair.get("number", 0)
    sql = pair.get("sql", "")
    if not sql.strip():
        r = {"db": f"db-{db_num}", "q": q_num, "ok": False, "rows": 0, "error": "empty sql"}
    else:
        rows, cols = run_sql(db_num, sql, limit=sql_limit)
        if rows is not None:
            r = {"db": f"db-{db_num}", "q": q_num, "ok": True, "rows": len(rows), "error": None}
        else:
            err = cols[:120] if isinstance(cols, str) else str(cols)[:120]
            err_full = str(cols) if isinstance(cols, str) else str(cols)
            r = {"db": f"db-{db_num}", "q": q_num, "ok": False, "rows": 0, "error": err, "error_full": err_full}
    try:
        _log_path.parent.mkdir(parents=True, exist_ok=True)
        _entry = {"timestamp": int(time.time() * 1000), "db": r["db"], "q": r["q"], "ok": r["ok"], "rows": r.get("rows", 0), "error": r.get("error")}
        open(_log_path, "a").write(json.dumps(_entry) + "\n")
    except Exception:
        pass
    return r

def test_all_databases(mounted: dict, db_nums: list = None, max_queries_per_db: int = None, max_workers: int = 8, sql_limit: int = 50, only_pairs: list = None) -> list:
    """Run all queries in parallel across all DBs; returns flat list of {db, q, ok, rows, error}.
    only_pairs: if set, list of (db_num, q_num) tuples to run only those queries (e.g. from last run's failures)."""
    db_nums = db_nums or DB_NUMS_TEST
    tasks = []
    want_set = {(n, q) for n, q in (only_pairs or [])}
    for n in db_nums:
        pairs = get_bird_pairs(mounted.get(n, {}), n)
        take = min(max_queries_per_db or len(pairs), len(pairs))
        for pair in pairs[:take]:
            if only_pairs is None or (n, pair.get("number")) in want_set:
                tasks.append((n, pair))
    if not tasks:
        return []
    results = []
    with ThreadPoolExecutor(max_workers=min(max_workers, len(tasks))) as ex:
        futures = {ex.submit(_test_one_query, n, pair, sql_limit): (n, pair) for n, pair in tasks}
        for fut in as_completed(futures):
            results.append(fut.result())
    return sorted(results, key=lambda r: (r["db"], r["q"]))

# Run all queries in parallel (requires Docker PostgreSQL on ports 5436–5451)
# max_queries_per_db=30 for full run; max_workers=8 to reduce timeouts; sql_limit=50 for faster execution.
# run_only_failed=True: run only queries that failed (uses failed_list from last run, or RECENT_FAILED_PAIRS).
# Update RECENT_FAILED_PAIRS after each run to re-test only the queries that still fail.
RECENT_FAILED_PAIRS = (
    [(15, 14)]  # db-15: rebate_adjusted_payback_years ambiguous
    + [(6, 2)]  # db-6: recursive CTE (anchor ST_CoveredBy only, hierarchy<2)
)
run_only_failed = False  # True: run only failed; False: run all queries
test_results = []
if docker_ok and len(containers) >= 1:
    only_pairs = None
    if run_only_failed:
        prev_failed = globals().get("failed_list", [])
        if prev_failed:
            only_pairs = [(int(r["db"].split("-")[1]), r["q"]) for r in prev_failed]
            print(f"Running only {len(only_pairs)} failed queries from last run...")
        else:
            only_pairs = RECENT_FAILED_PAIRS
            print(f"Running only {len(only_pairs)} failed queries (from RECENT_FAILED_PAIRS)...")
    if not run_only_failed or only_pairs:
        test_results = test_all_databases(mounted, max_queries_per_db=30, max_workers=8, sql_limit=50, only_pairs=only_pairs if run_only_failed else None)
    total = len(test_results)
    passed = sum(1 for r in test_results if r["ok"])
    failed_list = [r for r in test_results if not r["ok"]]
    excl_str = f" ({len(excluded_nums)} excluded via config)" if excluded_nums else ""
    print(f"Total: {total} queries | Passed: {passed} | Failed: {len(failed_list)}{excl_str}")
    if failed_list:
        print("\n--- Failures ---")
        for r in failed_list[:30]:
            print(f"  {r['db']} Q{r['q']}: {r['error']}")
        if len(failed_list) > 30:
            print(f"  ... and {len(failed_list) - 30} more")
    else:
        for r in test_results:
            status = "✓" if r["ok"] else "✗"
            msg = f"{r['rows']} rows" if r["ok"] else r["error"]
            print(f"  {r['db']} Q{r['q']}: {status} {msg}")
    if failed_list and any("does not exist" in str(r.get("error", "")) for r in failed_list):
        print("\n  Hint: If you see 'relation X does not exist', schema/data may not be loaded.")
        print("  Run: ./scripts/setup_docker.sh -a (from client/) or ./scripts/docker_postgres_qa.sh -a (from repo root)")
else:
    print("Docker not available or no PostgreSQL containers. See README.md for setup.")

Running only 1 failed queries from last run...
Total: 1 queries | Passed: 1 | Failed: 0 (3 excluded via config)
  db-6 Q2: ✓ 0 rows


In [17]:
# Debugging: rerun only the failed queries for each failed database, and print up to the first 3 failures per DB with full error detail.
# Identify all distinct database numbers that had any failed queries in test_results.
failed_db_nums = sorted({int(r["db"].split("-")[1]) for r in test_results if not r["ok"]})
if failed_db_nums:
    failed_db_results = test_all_databases(mounted, db_nums=failed_db_nums, max_queries_per_db=30, max_workers=4, sql_limit=50)
    failed_db_failures = [r for r in failed_db_results if not r["ok"]]
    print(f"Databases with failures: {failed_db_nums}")
    print(f"Total: {len(failed_db_failures)} failures of {len(failed_db_results)} total queries across failed DBs")
    for r in failed_db_failures[:30]:
        print(f"\n--- {r['db']} Q{r['q']} ---")
        print(r.get("error_full", r.get("error", "?"))[:500])
else:
    print("No failed databases to rerun.")

No failed databases to rerun.


In [18]:
# Summary table — shows pass/fail per query.
try:
    import pandas as pd
    _has_pandas = True
except Exception:
    _has_pandas = False

if test_results:
    if _has_pandas:
        df = pd.DataFrame(test_results)
        df["status"] = df["ok"].map(lambda x: "✓" if x else "✗")
        display(df[["db", "status", "rows", "error"]])
    else:
        print("pandas not installed. pip install pandas (or pip install -r requirements.txt)")
        for r in test_results[:20]:
            s = "✓" if r["ok"] else "✗"
            print(f"  {r['db']} Q{r['q']}: {s} {r.get('rows', 0) or r.get('error', '')}")
        if len(test_results) > 20:
            print(f"  ... and {len(test_results) - 20} more")

import pandas as pd
failed_df = pd.DataFrame([r for r in test_results if not r["ok"]])
if not failed_df.empty:
    print("Failed queries DataFrame:")
    display(failed_df[["db", "q", "error"]])
else:
    print("No failed queries.")

,db,status,rows,error
0,db-6,✓,0,None


No failed queries.


In [19]:
import pandas as pd
df = pd.DataFrame(test_results)
recent_failures = df[~df["ok"]].sort_values(by="q", ascending=False).head(5)
print("Recent failures:")
print(recent_failures[["db", "q", "error"]].to_string(index=False))

Recent failures:
Empty DataFrame
Columns: [db, q, error]
Index: []


In [20]:
import pandas as pd
failed_df = pd.DataFrame([r for r in test_results if not r["ok"]])
if not failed_df.empty:
    print("Failed queries DataFrame:")
    display(failed_df[["db", "q", "error"]])
else:
    print("No failed queries.")

No failed queries.
